# Modelle über Models-as-a-Service nutzen

Dieser AI-Namespace wurde über den Self-Service-Katalog bestellt. Der API-Key für MaaS steckt schon in der Workbench, als Umgebungsvariablen aus dem Secret `maas-zugang`. Nichts davon musst du selbst einrichten.

In [ ]:
import os, requests

base_url = os.environ["MAAS_BASE_URL"]
headers = {"Authorization": f"Bearer {os.environ['MAAS_API_KEY']}"}

# Welche Modelle darf dieser Namespace nutzen?
models = requests.get(f"{base_url}/models", headers=headers, timeout=30).json()
[m["id"] for m in models.get("data", [])]

## Eine Frage an ein Modell

Dieselbe OpenAI-kompatible API für das selbst gehostete Granite und für externe Modelle wie Claude Haiku. Nur der Modellname ändert sich.

In [ ]:
def frage(modell, text, max_tokens=200):
    r = requests.post(f"{base_url}/chat/completions", headers=headers, timeout=120, json={
        "model": modell,
        "messages": [{"role": "user", "content": text}],
        "max_tokens": max_tokens,
    })
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

print(frage(os.environ.get("MAAS_MODEL", "granite"), "Erkläre in zwei Sätzen, was Models-as-a-Service ist."))

## Mini-RAG: Antworten aus eigenem Text

Ohne Vektordatenbank, nur um das Prinzip zu zeigen: Wir geben dem Modell einen Ausschnitt aus einem internen Dokument mit und lassen es nur daraus antworten.

In [ ]:
dokument = """
Betriebshandbuch Plattform-Team (Auszug):
- Namespaces bestellt man im Ansible Automation Portal oder per Agent über MCP.
- Datenbanken kosten Geld und brauchen eine Freigabe in AAP.
- Edge-Cluster holen ihre Konfiguration selbst aus Git (Pull-Modell).
"""

print(frage(os.environ.get("MAAS_MODEL", "granite"),
            f"Beantworte nur anhand dieses Textes: Wie bekomme ich eine Datenbank?\n\n{dokument}"))